# Band power and memory load

Power in each load level, during encoding or retrieval.
Set `PHASE` to `"encoding"` or `"retrieval"` and `BAND` to `"theta"`, `"low gamma"` or `"high gamma"`.
The band limits are defined in `BANDS` in `helpers.py`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from helpers import (BANDS, load_data, bandpass_filter, level_windows, band_power,
                     level_tests, load_correlation, fdr, plot_levels)

PHASE = "encoding"               # "encoding" or "retrieval"
BAND = "theta"                   # "theta", "low gamma" or "high gamma"

DATA_DIR = "data"
PATIENT = "kh9_1"
SR = 1024                        # sampling rate (Hz)
LEVELS = ["Level1", "Level2", "Level3"]
COLORS = ["#FFA07A", "#FF4500", "#8B0000"]
ENCODING_POST = 1400             # samples kept after the last encoding event
RETRIEVAL_LENGTH = 1024          # samples from retrieval onset (1 s)
ALPHA = 0.05

OUT_DIR = Path("results") / PHASE / BAND.replace(" ", "_") / "power"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and filter

In [ ]:
data, events, channel_names = load_data(DATA_DIR, PATIENT)
filtered = bandpass_filter(data, *BANDS[BAND], SR)
print(BAND, BANDS[BAND], "Hz")

## 2. One window per level block

- Encoding: from the second-to-last encoding event to 1400 samples after the last one.
- Retrieval: 1 s from the retrieval onset.

In [ ]:
windows = level_windows(events, PHASE, LEVELS, ENCODING_POST, RETRIEVAL_LENGTH)
for level, w in windows.items():
    print(level, len(w), "trials")

## 3. Power

Area under the Welch power spectrum of the band-pass filtered signal, per trial and channel.

In [ ]:
power = band_power(filtered, windows, SR)       # level -> (trials, channels)

## 4. Statistics

Per channel: Friedman test across the three levels, Wilcoxon signed-rank tests for each pair of levels,
and Spearman correlation between load (1, 2, 3) and power. FDR correction across channels.

In [ ]:
p_friedman, p_pairs = level_tests(power)
rho, p_spearman = load_correlation(power)

table = pd.DataFrame({"channel": channel_names,
                      "friedman_p": p_friedman, "friedman_p_fdr": fdr(p_friedman),
                      "spearman_rho": rho, "spearman_p": p_spearman, "spearman_p_fdr": fdr(p_spearman)})
for (a, b), p in p_pairs.items():
    table[f"{a}_vs_{b}_p"] = p
    table[f"{a}_vs_{b}_p_fdr"] = fdr(p)
table.to_csv(OUT_DIR / "stats.csv", index=False)

significant = np.where(p_friedman < ALPHA)[0]
print("Friedman p < 0.05:  ", ", ".join(channel_names[significant]))
print("Friedman FDR < 0.05:", ", ".join(channel_names[fdr(p_friedman) < ALPHA]))
print("Spearman FDR < 0.05:", ", ".join(channel_names[fdr(p_spearman) < ALPHA]))

In [ ]:
table.iloc[significant]

## 5. Plots

Boxplots for channels with Friedman p < 0.05. Stars show the uncorrected Wilcoxon p-values.

In [ ]:
labels = ["Level 1", "Level 2", "Level 3"]

for ch in significant:
    pair_p = {(LEVELS.index(a), LEVELS.index(b)): p[ch] for (a, b), p in p_pairs.items()}
    plot_levels([power[level][:, ch] for level in LEVELS], labels, COLORS, pair_p,
                ylabel="Power (AUC, µV²/Hz)", title=f"{channel_names[ch]} – {PHASE}, {BAND}",
                save_path=OUT_DIR / f"{channel_names[ch]}.png")